# SHALE paper: deep-learning length side-channel attack

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100). Add Input → search and add **CREMA-D** (e.g. `ejlok1/cremad`) and **Google Speech Commands** (e.g. `yashdogra/speech-commands` or `neehakurelli/google-speech-commands`).

Then **Run All**. Expected time about 3–5 hours on a T4. Results are saved after every experiment to `/kaggle/working/shale_dl_results.json`. If the session stops, just run again: finished experiments are skipped. Download that JSON file at the end and send it back.

For a 5-minute smoke test first, set `QUICK = True` in Cell 3.

In [ ]:
# Cell 1: write the SHALE coder and ECLQ modules (identical to the paper's implementation)
import os
os.makedirs('/kaggle/working/shale_mod', exist_ok=True)
open('/kaggle/working/shale_mod/codec.py','w').write('"""\nSHALE lossless audio codec (layer 1 of SHALE).\n\nFrame-independent integer linear prediction (fixed polynomial predictors of\norder 0-4 and quantised LPC of order 1..P) with partitioned Rice coding of the\nresidual and inter-channel (left/side, right/side, mid/side) decorrelation.\nThe design follows the well-known FLAC/Shorten family (RFC 9639); the bitstream\nis SHALE\'s own and every frame can be decoded on its own.\n"""\nimport numpy as np\nfrom numba import njit\n\n# --------------------------------------------------------------------------\n# bit I/O (state: [byte position, accumulator, bits in accumulator])\n# --------------------------------------------------------------------------\n\n\n@njit(cache=True, inline="always")\ndef _put(buf, st, v, n):\n    if n == 0:\n        return\n    acc = (st[1] << np.uint64(n)) | (np.uint64(v) & ((np.uint64(1) << np.uint64(n)) - np.uint64(1)))\n    nb = st[2] + np.uint64(n)\n    while nb >= np.uint64(8):\n        nb -= np.uint64(8)\n        buf[st[0]] = np.uint8((acc >> nb) & np.uint64(0xFF))\n        st[0] += np.uint64(1)\n    st[1] = acc & ((np.uint64(1) << nb) - np.uint64(1))\n    st[2] = nb\n\n\n@njit(cache=True)\ndef _flush(buf, st):\n    if st[2] > np.uint64(0):\n        _put(buf, st, 0, int(np.uint64(8) - st[2]))\n\n\n@njit(cache=True, inline="always")\ndef _get(buf, st, n):\n    # st: [bit position]\n    v = np.uint64(0)\n    for _ in range(n):\n        p = st[0]\n        bit = (np.uint64(buf[p >> np.uint64(3)]) >> (np.uint64(7) - (p & np.uint64(7)))) & np.uint64(1)\n        v = (v << np.uint64(1)) | bit\n        st[0] = p + np.uint64(1)\n    return v\n\n\n@njit(cache=True, inline="always")\ndef _get_fast(buf, st, n):\n    """read n<=32 bits using byte steps"""\n    if n == 0:\n        return np.uint64(0)\n    p = st[0]\n    byte = p >> np.uint64(3)\n    off = p & np.uint64(7)\n    need = off + np.uint64(n)\n    v = np.uint64(0)\n    nbytes = (need + np.uint64(7)) >> np.uint64(3)\n    for i in range(int(nbytes)):\n        idx = byte + np.uint64(i)\n        b = np.uint64(buf[idx]) if idx < np.uint64(buf.shape[0]) else np.uint64(0)\n        v = (v << np.uint64(8)) | b\n    v = (v >> (nbytes * np.uint64(8) - need)) & ((np.uint64(1) << np.uint64(n)) - np.uint64(1))\n    st[0] = p + np.uint64(n)\n    return v\n\n\n@njit(cache=True, inline="always")\ndef _get_unary(buf, st):\n    q = 0\n    while True:\n        p = st[0]\n        bit = (buf[p >> np.uint64(3)] >> (7 - (p & np.uint64(7)))) & 1\n        st[0] = p + np.uint64(1)\n        if bit:\n            return q\n        q += 1\n\n\n@njit(cache=True, inline="always")\ndef _signed(v, n):\n    v = np.int64(v)\n    if v >= (np.int64(1) << (n - 1)):\n        v -= np.int64(1) << n\n    return v\n\n\n@njit(cache=True, inline="always")\ndef _zig(r):\n    return np.uint64((r << 1) ^ (r >> 63))\n\n\n@njit(cache=True, inline="always")\ndef _unzig(u):\n    return np.int64(u >> np.uint64(1)) ^ -np.int64(u & np.uint64(1))\n\n# --------------------------------------------------------------------------\n# prediction\n# --------------------------------------------------------------------------\n\n\n@njit(cache=True)\ndef _fixed_residual(x, order, out):\n    n = x.shape[0]\n    for i in range(order):\n        out[i] = 0\n    for i in range(order, n):\n        if order == 0:\n            out[i] = x[i]\n        elif order == 1:\n            out[i] = x[i] - x[i - 1]\n        elif order == 2:\n            out[i] = x[i] - 2 * x[i - 1] + x[i - 2]\n        elif order == 3:\n            out[i] = x[i] - 3 * x[i - 1] + 3 * x[i - 2] - x[i - 3]\n        else:\n            out[i] = x[i] - 4 * x[i - 1] + 6 * x[i - 2] - 4 * x[i - 3] + x[i - 4]\n\n\n@njit(cache=True)\ndef _lpc_residual(x, coef, order, shift, out):\n    n = x.shape[0]\n    for i in range(order):\n        out[i] = 0\n    for i in range(order, n):\n        s = np.int64(0)\n        for j in range(order):\n            s += coef[j] * x[i - 1 - j]\n        out[i] = x[i] - (s >> shift)\n\n\n@njit(cache=True)\ndef _levinson(r, maxp, lpc):\n    """lpc[p-1, :p] = predictor coefficients of order p (x[n] ~ sum a_j x[n-1-j])"""\n    err = r[0]\n    errs = np.zeros(maxp + 1)\n    errs[0] = err\n    a = np.zeros(maxp + 1)\n    tmp = np.zeros(maxp + 1)\n    for p in range(1, maxp + 1):\n        if err <= 0:\n            for q in range(p, maxp + 1):\n                lpc[q - 1, :q] = 0.0\n                errs[q] = 0.0\n            return errs\n        acc = r[p]\n        for j in range(1, p):\n            acc -= a[j] * r[p - j]\n        k = acc / err\n        tmp[:p] = a[:p]\n        a[p] = k\n        for j in range(1, p):\n            a[j] = tmp[j] - k * tmp[p - j]\n        err *= (1.0 - k * k)\n        for j in range(p):\n            lpc[p - 1, j] = a[j + 1]\n        errs[p] = err\n    return errs\n\n\n@njit(cache=True)\ndef _quantise(a, order, prec, qc):\n    cmax = 0.0\n    for j in range(order):\n        if abs(a[j]) > cmax:\n            cmax = abs(a[j])\n    if cmax <= 0.0:\n        return -1\n    e = int(np.floor(np.log2(cmax))) + 1      # cmax < 2^e\n    shift = prec - 1 - e\n    if shift > 20:\n        shift = 20\n    if shift < 0:\n        return -1\n    lim = (1 << (prec - 1)) - 1\n    err = 0.0\n    for j in range(order):\n        err += a[j] * (1 << shift)\n        q = int(np.round(err))\n        if q > lim:\n            q = lim\n        if q < -lim - 1:\n            q = -lim - 1\n        qc[j] = q\n        err -= q\n    return shift\n\n# --------------------------------------------------------------------------\n# residual cost estimation and Rice coding\n# --------------------------------------------------------------------------\n\n\n@njit(cache=True)\ndef _rice_k(s, n):\n    # optimal-ish Rice parameter for n values with sum s\n    if n <= 0 or s <= 0:\n        return 0\n    m = s / n\n    k = 0\n    while k < 30 and (np.uint64(1) << np.uint64(k + 1)) <= m + 1:\n        k += 1\n    return k\n\n\n@njit(cache=True)\ndef _best_partition(res, order, maxpo, n):\n    """returns (estimated bits, partition order)"""\n    # prefix sums of zigzag residuals\n    ps = np.zeros(n + 1, np.uint64)\n    for i in range(n):\n        if i < order:\n            ps[i + 1] = ps[i]\n        else:\n            ps[i + 1] = ps[i] + _zig(res[i])\n    best = np.int64(1) << 62\n    bpo = 0\n    for po in range(maxpo + 1):\n        np_ = 1 << po\n        if n % np_ != 0:\n            break\n        plen = n // np_\n        if plen <= order:\n            break\n        bits = np.int64(4)\n        for p in range(np_):\n            a = p * plen\n            b = a + plen\n            cnt = plen - order if p == 0 else plen\n            s = ps[b] - ps[a]\n            k = _rice_k(s, cnt)\n            bits += 5 + cnt * (k + 1) + np.int64(s >> np.uint64(k))\n        if bits < best:\n            best = bits\n            bpo = po\n    return best, bpo\n\n\n@njit(cache=True)\ndef _write_residual(buf, st, res, order, po, n):\n    _put(buf, st, po, 4)\n    np_ = 1 << po\n    plen = n // np_\n    for p in range(np_):\n        a = p * plen + (order if p == 0 else 0)\n        b = (p + 1) * plen\n        s = np.uint64(0)\n        mx = np.uint64(0)\n        for i in range(a, b):\n            u = _zig(res[i])\n            s += u\n            if u > mx:\n                mx = u\n        k = _rice_k(s, b - a)\n        # refine k exactly among k-1, k, k+1\n        bestk = k\n        bestc = np.int64(1) << 62\n        for kk in range(max(0, k - 1), min(30, k + 1) + 1):\n            c = np.int64(0)\n            for i in range(a, b):\n                c += np.int64(_zig(res[i]) >> np.uint64(kk)) + kk + 1\n            if c < bestc:\n                bestc = c\n                bestk = kk\n        # escape if unary part could be huge\n        w = 0\n        while w < 40 and (np.uint64(1) << np.uint64(w)) <= mx:\n            w += 1\n        if np.int64(w) * (b - a) + 6 < bestc:\n            _put(buf, st, 31, 5)\n            _put(buf, st, w, 6)\n            for i in range(a, b):\n                _put(buf, st, _zig(res[i]), w)\n        else:\n            _put(buf, st, bestk, 5)\n            kk = np.uint64(bestk)\n            for i in range(a, b):\n                u = _zig(res[i])\n                q = u >> kk\n                while q >= np.uint64(32):\n                    _put(buf, st, 0, 32)\n                    q -= np.uint64(32)\n                _put(buf, st, 1, int(q) + 1)\n                _put(buf, st, u & ((np.uint64(1) << kk) - np.uint64(1)), bestk)\n\n\n@njit(cache=True)\ndef _read_residual(buf, st, res, order, n):\n    po = int(_get_fast(buf, st, 4))\n    np_ = 1 << po\n    plen = n // np_\n    for p in range(np_):\n        a = p * plen + (order if p == 0 else 0)\n        b = (p + 1) * plen\n        k = int(_get_fast(buf, st, 5))\n        if k == 31:\n            w = int(_get_fast(buf, st, 6))\n            for i in range(a, b):\n                res[i] = _unzig(_get_fast(buf, st, w))\n        else:\n            kk = np.uint64(k)\n            for i in range(a, b):\n                q = np.uint64(_get_unary(buf, st))\n                lo = _get_fast(buf, st, k)\n                res[i] = _unzig((q << kk) | lo)\n\n# --------------------------------------------------------------------------\n# subframe encode / decode\n# --------------------------------------------------------------------------\n# types: 0 verbatim, 1 constant, 2 fixed, 3 lpc\n\n\n@njit(cache=True)\ndef _encode_sub(buf, st, x, bits, maxp, prec, maxpo, res, best_res, qc, lpcmat, win, dry, exhaustive):\n    n = x.shape[0]\n    # constant?\n    const = True\n    for i in range(1, n):\n        if x[i] != x[0]:\n            const = False\n            break\n    if const:\n        if not dry:\n            _put(buf, st, 1, 3)\n            _put(buf, st, np.uint64(x[0]) & ((np.uint64(1) << np.uint64(bits)) - np.uint64(1)), bits)\n        return 3 + bits\n    best = np.int64(3) + np.int64(bits) * n   # verbatim\n    btype = 0\n    border = 0\n    bpo = 0\n    bshift = 0\n    # fixed predictors: pick order by sum |e| (one pass), then exact cost for the best\n    e = np.zeros(5)\n    for i in range(4, n):\n        d0 = x[i]; d1 = d0 - x[i - 1]; d2 = d1 - (x[i - 1] - x[i - 2])\n        d3 = d2 - (x[i - 1] - 2 * x[i - 2] + x[i - 3])\n        d4 = d3 - (x[i - 1] - 3 * x[i - 2] + 3 * x[i - 3] - x[i - 4])\n        e[0] += abs(d0); e[1] += abs(d1); e[2] += abs(d2); e[3] += abs(d3); e[4] += abs(d4)\n    fo = 0\n    for o in range(1, 5):\n        if e[o] < e[fo]:\n            fo = o\n    if n <= 4 or exhaustive:\n        olist = np.arange(0, min(5, n))\n    else:\n        olist = np.array([fo])\n    for order in olist:\n        _fixed_residual(x, order, res)\n        c, po = _best_partition(res, order, maxpo, n)\n        c += 3 + 3 + order * bits\n        if c < best:\n            best = c\n            btype = 2\n            border = order\n            bpo = po\n            best_res[:] = res[:]\n    # LPC\n    pmax = min(maxp, n - 1)\n    if pmax >= 1:\n        r = np.zeros(pmax + 1)\n        xw = np.empty(n)\n        for i in range(n):\n            xw[i] = x[i] * win[i]\n        for lag in range(pmax + 1):\n            s = 0.0\n            for i in range(lag, n):\n                s += xw[i] * xw[i - lag]\n            r[lag] = s\n        if r[0] > 0:\n            r[0] *= 1.0 + 1e-9\n            errs = _levinson(r, pmax, lpcmat)\n            if exhaustive:\n                orders = np.arange(1, pmax + 1)\n            else:\n                # estimated bits per order from the Levinson prediction error\n                be = 1e300\n                bo = 1\n                for o in range(1, pmax + 1):\n                    ev = errs[o] / n\n                    bps = 0.5 * np.log2(ev) if ev > 1e-12 else 0.0\n                    if bps < 0:\n                        bps = 0.0\n                    est = n * bps + o * (prec + bits)\n                    if est < be:\n                        be = est\n                        bo = o\n                orders = np.array([bo])\n            for order in orders:\n                shift = _quantise(lpcmat[order - 1], order, prec, qc)\n                if shift < 0:\n                    continue\n                _lpc_residual(x, qc, order, shift, res)\n                c, po = _best_partition(res, order, maxpo, n)\n                c += 3 + 5 + 4 + 5 + order * prec + order * bits\n                if c < best:\n                    best = c\n                    btype = 3\n                    border = order\n                    bpo = po\n                    bshift = shift\n                    best_res[:] = res[:]\n                    # remember coefficients\n                    for j in range(order):\n                        lpcmat[pmax, j] = qc[j]\n    if dry:\n        return best\n    mask = (np.uint64(1) << np.uint64(bits)) - np.uint64(1)\n    if btype == 0:\n        _put(buf, st, 0, 3)\n        for i in range(n):\n            _put(buf, st, np.uint64(x[i]) & mask, bits)\n    elif btype == 2:\n        _put(buf, st, 2, 3)\n        _put(buf, st, border, 3)\n        for i in range(border):\n            _put(buf, st, np.uint64(x[i]) & mask, bits)\n        _write_residual(buf, st, best_res, border, bpo, n)\n    else:\n        _put(buf, st, 3, 3)\n        _put(buf, st, border, 5)\n        _put(buf, st, prec - 1, 4)\n        _put(buf, st, bshift, 5)\n        pm = (np.uint64(1) << np.uint64(prec)) - np.uint64(1)\n        for j in range(border):\n            _put(buf, st, np.uint64(np.int64(lpcmat[pmax, j])) & pm, prec)\n        for i in range(border):\n            _put(buf, st, np.uint64(x[i]) & mask, bits)\n        _write_residual(buf, st, best_res, border, bpo, n)\n    return best\n\n\n@njit(cache=True)\ndef _decode_sub(buf, st, x, bits, res):\n    n = x.shape[0]\n    t = int(_get_fast(buf, st, 3))\n    if t == 1:\n        v = _signed(_get_fast(buf, st, bits), bits)\n        for i in range(n):\n            x[i] = v\n    elif t == 0:\n        for i in range(n):\n            x[i] = _signed(_get_fast(buf, st, bits), bits)\n    elif t == 2:\n        order = int(_get_fast(buf, st, 3))\n        for i in range(order):\n            x[i] = _signed(_get_fast(buf, st, bits), bits)\n        _read_residual(buf, st, res, order, n)\n        for i in range(order, n):\n            if order == 0:\n                x[i] = res[i]\n            elif order == 1:\n                x[i] = res[i] + x[i - 1]\n            elif order == 2:\n                x[i] = res[i] + 2 * x[i - 1] - x[i - 2]\n            elif order == 3:\n                x[i] = res[i] + 3 * x[i - 1] - 3 * x[i - 2] + x[i - 3]\n            else:\n                x[i] = res[i] + 4 * x[i - 1] - 6 * x[i - 2] + 4 * x[i - 3] - x[i - 4]\n    else:\n        order = int(_get_fast(buf, st, 5))\n        prec = int(_get_fast(buf, st, 4)) + 1\n        shift = int(_get_fast(buf, st, 5))\n        coef = np.empty(order, np.int64)\n        for j in range(order):\n            coef[j] = _signed(_get_fast(buf, st, prec), prec)\n        for i in range(order):\n            x[i] = _signed(_get_fast(buf, st, bits), bits)\n        _read_residual(buf, st, res, order, n)\n        for i in range(order, n):\n            s = np.int64(0)\n            for j in range(order):\n                s += coef[j] * x[i - 1 - j]\n            x[i] = res[i] + (s >> shift)\n\n# --------------------------------------------------------------------------\n# frame encode / decode\n# --------------------------------------------------------------------------\n\n\n@njit(cache=True)\ndef _tukey(n):\n    w = np.ones(n)\n    a = 0.5\n    m = int(a * (n - 1) / 2)\n    for i in range(m + 1):\n        v = 0.5 * (1 - np.cos(2 * np.pi * i / (a * (n - 1)))) if n > 1 else 1.0\n        w[i] = v\n        w[n - 1 - i] = v\n    return w\n\n\n@njit(cache=True)\ndef encode_frame(frame, bits, maxp, prec, maxpo, out, exhaustive):\n    """frame: (n, C) int64. Writes into out (uint8); returns number of bytes."""\n    n, C = frame.shape\n    st = np.zeros(3, np.uint64)\n    res = np.zeros(n, np.int64)\n    bres = np.zeros(n, np.int64)\n    qc = np.zeros(33, np.int64)\n    lpcmat = np.zeros((max(maxp, 1) + 1, 33))\n    win = _tukey(n)\n    if C == 2:\n        L = frame[:, 0].copy()\n        R = frame[:, 1].copy()\n        S = L - R\n        M = (L + R) >> 1\n        cL = _encode_sub(out, st, L, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, True, exhaustive)\n        cR = _encode_sub(out, st, R, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, True, exhaustive)\n        cS = _encode_sub(out, st, S, bits + 1, maxp, prec, maxpo, res, bres, qc, lpcmat, win, True, exhaustive)\n        cM = _encode_sub(out, st, M, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, True, exhaustive)\n        costs = np.array([cL + cR, cL + cS, cR + cS, cM + cS])\n        mode = int(np.argmin(costs))\n        _put(out, st, mode, 2)\n        if mode == 0:\n            _encode_sub(out, st, L, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n            _encode_sub(out, st, R, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n        elif mode == 1:\n            _encode_sub(out, st, L, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n            _encode_sub(out, st, S, bits + 1, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n        elif mode == 2:\n            _encode_sub(out, st, R, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n            _encode_sub(out, st, S, bits + 1, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n        else:\n            _encode_sub(out, st, M, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n            _encode_sub(out, st, S, bits + 1, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n    else:\n        for c in range(C):\n            x = frame[:, c].copy()\n            _encode_sub(out, st, x, bits, maxp, prec, maxpo, res, bres, qc, lpcmat, win, False, exhaustive)\n    _flush(out, st)\n    return int(st[0])\n\n\n@njit(cache=True)\ndef decode_frame(buf, n, C, bits, out):\n    """out: (n, C) int64"""\n    st = np.zeros(1, np.uint64)\n    res = np.zeros(n, np.int64)\n    if C == 2:\n        mode = int(_get_fast(buf, st, 2))\n        a = np.zeros(n, np.int64)\n        b = np.zeros(n, np.int64)\n        if mode == 0:\n            _decode_sub(buf, st, a, bits, res)\n            _decode_sub(buf, st, b, bits, res)\n            out[:, 0] = a\n            out[:, 1] = b\n        elif mode == 1:\n            _decode_sub(buf, st, a, bits, res)\n            _decode_sub(buf, st, b, bits + 1, res)\n            out[:, 0] = a\n            out[:, 1] = a - b\n        elif mode == 2:\n            _decode_sub(buf, st, a, bits, res)\n            _decode_sub(buf, st, b, bits + 1, res)\n            out[:, 1] = a\n            out[:, 0] = a + b\n        else:\n            _decode_sub(buf, st, a, bits, res)   # mid\n            _decode_sub(buf, st, b, bits + 1, res)  # side\n            for i in range(n):\n                m = (a[i] << 1) | (b[i] & 1)\n                out[i, 0] = (m + b[i]) >> 1\n                out[i, 1] = (m - b[i]) >> 1\n    else:\n        for c in range(C):\n            a = np.zeros(n, np.int64)\n            _decode_sub(buf, st, a, bits, res)\n            out[:, c] = a\n    return int((st[0] + np.uint64(7)) >> np.uint64(3))\n\n\ndef max_frame_bytes(n, C, bits):\n    # verbatim worst case + headers, side channel +1 bit\n    return (n * C * (bits + 1) + 64 * C + 16) // 8 + 8\n\n\ndef encode_frames(x, bits, frame_len, maxp=12, prec=12, maxpo=8, exhaustive=False):\n    """x: (N, C) integer array. Returns list of bytes objects, one per frame."""\n    x = np.asarray(x, dtype=np.int64)\n    if x.ndim == 1:\n        x = x[:, None]\n    N, C = x.shape\n    out = np.zeros(max_frame_bytes(frame_len, C, bits), np.uint8)\n    frames = []\n    for s in range(0, N, frame_len):\n        fr = np.ascontiguousarray(x[s:s + frame_len])\n        nb = encode_frame(fr, bits, maxp, prec, maxpo, out, exhaustive)\n        frames.append(out[:nb].tobytes())\n    return frames\n\n\ndef decode_frames(frames, N, C, bits, frame_len):\n    x = np.zeros((N, C), np.int64)\n    for i, fb in enumerate(frames):\n        s = i * frame_len\n        n = min(frame_len, N - s)\n        buf = np.frombuffer(fb + b"\\x00" * 8, np.uint8)\n        out = np.zeros((n, C), np.int64)\n        decode_frame(buf, n, C, bits, out)\n        x[s:s + n] = out\n    return x\n')
open('/kaggle/working/shale_mod/eclq.py','w').write('"""\nEntropy-Constrained Length Quantisation (ECLQ) - SHALE layer 2.\n\nGiven the public distribution of compressed record lengths for one audio profile,\nchoose bucket sizes b_1 < ... < b_K (b_K = the uncompressed worst case) that minimise\n\n        E[b(l) - l]  +  lambda * H(B)        subject to  K <= Kmax,\n\nwhere b(l) is the smallest bucket >= l and B is the bucket index. The objective\ndecomposes over contiguous buckets, so an exact dynamic program over a grid of\ncandidate bucket tops finds the optimum. lambda = 0 gives the minimum-overhead\nK-bucket quantiser; a bisection on lambda meets a target entropy (leakage) budget.\n"""\nimport numpy as np\nfrom numba import njit\n\n\n@njit(cache=True)\ndef _dp(tops, mass, lsum, kmax, lam):\n    """tops: (M,) candidate tops (ascending, last = forced top).\n    mass[m], lsum[m]: probability mass and sum of l*p(l) of lengths in (tops[m-1], tops[m]].\n    Returns choice table for reconstruction: best cost per k, and back pointers."""\n    M = tops.shape[0]\n    W = np.zeros(M + 1)\n    S = np.zeros(M + 1)\n    for m in range(M):\n        W[m + 1] = W[m] + mass[m]\n        S[m + 1] = S[m] + lsum[m]\n    INF = 1e300\n    D = np.full((kmax + 1, M + 1), INF)\n    back = np.full((kmax + 1, M + 1), -1, np.int64)\n    D[0, 0] = 0.0\n    for k in range(1, kmax + 1):\n        for j in range(1, M + 1):\n            # bucket covers cells i..j-1 with top tops[j-1]\n            # skip empty top cells except the forced last one (no gain from empty buckets)\n            if mass[j - 1] <= 0.0 and j != M:\n                continue\n            best = INF\n            bi = -1\n            for i in range(0, j):\n                if D[k - 1, i] >= INF:\n                    continue\n                q = W[j] - W[i]\n                pad = q * tops[j - 1] - (S[j] - S[i])\n                ent = -q * np.log2(q) if q > 0 else 0.0\n                c = D[k - 1, i] + pad + lam * ent\n                if c < best:\n                    best = c\n                    bi = i\n            D[k, j] = best\n            back[k, j] = bi\n    return D, back\n\n\ndef _cells(lengths, weights, top, granule):\n    lengths = np.asarray(lengths, np.int64)\n    if weights is None:\n        weights = np.ones(len(lengths))\n    weights = np.asarray(weights, float) / np.sum(weights)\n    g = max(1, int(granule))\n    ntop = int(np.ceil(top / g))\n    tops = np.minimum(np.arange(1, ntop + 1) * g, top).astype(np.int64)\n    tops[-1] = top\n    cell = np.clip(np.ceil(lengths / g).astype(np.int64) - 1, 0, ntop - 1)\n    mass = np.bincount(cell, weights=weights, minlength=ntop)\n    lsum = np.bincount(cell, weights=weights * lengths, minlength=ntop)\n    return tops, mass, lsum\n\n\ndef design(lengths, top, kmax, lam=0.0, granule=16, weights=None):\n    """Return bucket tops (ascending, last == top) minimising overhead + lam*H with <= kmax buckets."""\n    assert np.max(lengths) <= top\n    tops, mass, lsum = _cells(lengths, weights, top, granule)\n    D, back = _dp(tops.astype(float), mass, lsum, kmax, float(lam))\n    M = len(tops)\n    k = int(np.argmin(D[1:, M])) + 1\n    # reconstruct\n    b = []\n    j = M\n    while k > 0:\n        b.append(int(tops[j - 1]))\n        j = int(back[k, j])\n        k -= 1\n    return np.array(sorted(b), np.int64)\n\n\ndef quantise(lengths, buckets):\n    """bucket index and padded size for each length"""\n    idx = np.searchsorted(buckets, lengths, side="left")\n    return idx, buckets[idx]\n\n\ndef stats(lengths, buckets, weights=None):\n    """expected overhead (bytes and relative), bucket entropy H(B) in bits, K"""\n    lengths = np.asarray(lengths, np.int64)\n    w = np.ones(len(lengths)) if weights is None else np.asarray(weights, float)\n    w = w / w.sum()\n    idx, padded = quantise(lengths, buckets)\n    p = np.bincount(idx, weights=w, minlength=len(buckets))\n    p = p[p > 0]\n    H = float(-(p * np.log2(p)).sum())\n    return dict(pad_bytes=float((w * (padded - lengths)).sum()),\n                mean_len=float((w * lengths).sum()),\n                mean_padded=float((w * padded).sum()),\n                H=H, K=len(buckets), log2K=float(np.log2(len(buckets))))\n\n\ndef _reconstruct(tops, back, k, M):\n    b = []\n    j = M\n    while k > 0:\n        b.append(int(tops[j - 1]))\n        j = int(back[k, j])\n        k -= 1\n    return np.array(sorted(b), np.int64)\n\n\ndef design_entropy(lengths, top, kmax, h_target, granule=16, weights=None, lams=None):\n    """Least-overhead bucket set with profile entropy H(B) <= h_target and K <= kmax.\n    One DP per lambda yields the optimum for every bucket count k <= kmax at once;\n    sweeping lambda over a geometric grid and k = 1..kmax traces the attainable\n    (entropy, overhead) frontier, from which the best feasible point is taken."""\n    tops, mass, lsum = _cells(lengths, weights, top, granule)\n    M = len(tops)\n    if lams is None:\n        lams = np.concatenate([[0.0], np.geomspace(1e-2, 1e6, 48)]) * top / 1000.0\n    best, bestc = np.array([top], np.int64), float(top)\n    for lam in lams:\n        D, back = _dp(tops.astype(float), mass, lsum, kmax, float(lam))\n        for k in range(1, kmax + 1):\n            if D[k, M] >= 1e299:\n                continue\n            b = _reconstruct(tops, back, k, M)\n            st = stats(lengths, b, weights)\n            if st["H"] <= h_target + 1e-12 and st["mean_padded"] < bestc:\n                best, bestc = b, st["mean_padded"]\n    return best\n\n\ndef pow2_buckets(top, lmin=16):\n    """baseline: power-of-two buckets up to top"""\n    b = []\n    v = lmin\n    while v < top:\n        b.append(v)\n        v *= 2\n    b.append(top)\n    return np.array(b, np.int64)\n\n\ndef uniform_buckets(top, k):\n    """baseline: K equal-width buckets"""\n    return np.unique(np.ceil(np.arange(1, k + 1) * top / k).astype(np.int64))\n')
print('modules written')


In [ ]:
# Cell 2: bucket tables designed on the public VCTK profile, and the in-domain design actors
CONST = {"buckets": {"320": {"K2": [424, 644], "K4": [236, 380, 468, 644], "H0.1": [492, 644], "H0.3": [472, 532, 564, 644]}, "1600": {"K2": [2096, 3204], "K4": [1152, 1872, 2272, 3204], "H0.1": [2416, 2688, 2704, 3204], "H0.3": [2240, 3204]}}, "design_actors": ["1001", "1004", "1005", "1007", "1010", "1012", "1013", "1018", "1020", "1022", "1023", "1025", "1027", "1028", "1030", "1031", "1032", "1033", "1035", "1036", "1039", "1040", "1042", "1043", "1044", "1045", "1047", "1048", "1050", "1055", "1056", "1057", "1058", "1059", "1063", "1070", "1071", "1073", "1076", "1077", "1078", "1081", "1082", "1083", "1091"], "file_ratios": {"K2": [0.63, 1.0], "K4": [0.525, 0.63, 0.762, 1.0], "K8": [0.478, 0.519, 0.561, 0.63, 0.687, 0.736, 0.806, 1.0]}}


In [ ]:
# ======================================================================
# SHALE paper - deep-learning length side-channel attack (Kaggle, GPU)
# Inputs (Add Input in Kaggle): CREMA-D audio and Google Speech Commands.
# Output: /kaggle/working/shale_dl_results.json  (send this file back)
# ======================================================================
import os, re, glob, json, time, math, random, sys
import numpy as np
import soundfile as sf
import torch, torch.nn as nn, torch.nn.functional as Fnn
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from concurrent.futures import ProcessPoolExecutor

sys.path.insert(0, '/kaggle/working/shale_mod')
import codec, eclq

QUICK = False          # True = small smoke test (few files, 2 epochs)
NFOLD = 5
EPOCHS = 40
OUT = '/kaggle/working/shale_dl_results.json'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', DEV)
C = CONST
torch.manual_seed(0); np.random.seed(0); random.seed(0)

# ---------------------------------------------------------------- data discovery
allwav = glob.glob('/kaggle/input/**/*.wav', recursive=True)
crema = sorted({os.path.basename(f): f for f in allwav if re.match(r'^\d{4}_[A-Z]{3}_[A-Z]{3}_[A-Z]{2}\.wav$', os.path.basename(f))}.values())
sc = [f for f in allwav if re.match(r'^[0-9a-f]{8}_nohash_\d+\.wav$', os.path.basename(f))
      and '_background_noise_' not in f]
print('CREMA-D files', len(crema), ' Speech Commands files', len(sc))

def record_lengths(f, frames=(320, 1600, 4096)):
    x, sr = sf.read(f, dtype='int16')
    if x.ndim > 1: x = x[:, 0]
    if sr != 16000: return None
    x = x.astype(np.int64)[:, None]
    out = dict(N=len(x))
    for F in frames:
        fr = codec.encode_frames(x, 16, F)
        L = []
        for i, fb in enumerate(fr):
            raw = 2 * min(F, len(x) - i * F)
            L.append(4 + (raw if len(fb) >= raw else len(fb)))     # verbatim fallback
        out[F] = np.array(L, np.int32)
    return out

def lengths_for(files):
    codec.encode_frames(np.zeros((4000, 1), np.int64), 16, 320)   # JIT warm-up
    with ProcessPoolExecutor(os.cpu_count()) as ex:
        return list(ex.map(record_lengths, files, chunksize=32))

# ---------------------------------------------------------------- padding schemes
def padme(L):
    L = np.asarray(L, np.int64); out = L.copy()
    for i, l in enumerate(L):
        if l < 2: continue
        E = int(math.floor(math.log2(l))); S = int(math.floor(math.log2(E))) + 1 if E > 0 else 1
        z = max(E - S, 0); m = (1 << z) - 1
        out[i] = (l + m) & ~m
    return out

def schemes(F, top, indomain_prof):
    B = {k: np.array(v) for k, v in C['buckets'][str(F)].items()}
    S = {'none': lambda L: L,
         'padme': lambda L: np.minimum(padme(L), top),
         'pow2': lambda L, b=eclq.pow2_buckets(top): b[np.searchsorted(b, L)]}
    for k in ('K2', 'K4', 'H0.1', 'H0.3'):
        S['eclq_' + k] = (lambda L, b=B[k]: b[np.searchsorted(b, L)])
    if indomain_prof is not None:
        b = eclq.design_entropy(indomain_prof, top, 4, 0.1, 4 if F <= 400 else 16)
        S['eclq_H0.1_indomain'] = (lambda L, b=b: b[np.searchsorted(b, L)])
    return S

# ---------------------------------------------------------------- models
class CNN(nn.Module):
    def __init__(s, ncls):
        super().__init__()
        s.c = nn.Sequential(nn.Conv1d(2, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
                            nn.Conv1d(64, 128, 5, padding=2), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
                            nn.Conv1d(128, 128, 5, padding=2, dilation=1), nn.BatchNorm1d(128), nn.ReLU(),
                            nn.Conv1d(128, 128, 3, padding=2, dilation=2), nn.BatchNorm1d(128), nn.ReLU())
        s.h = nn.Sequential(nn.Linear(256 + 2, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, ncls))
    def forward(s, x, m, d):
        z = s.c(torch.stack([x, m], 1))
        mm = Fnn.max_pool1d(m[:, None], 2) if z.shape[-1] != m.shape[-1] else m[:, None]
        mm = mm[..., :z.shape[-1]]
        avg = (z * mm).sum(-1) / mm.sum(-1).clamp(min=1)
        mx = (z - 1e4 * (1 - mm)).max(-1).values
        return s.h(torch.cat([avg, mx, d], 1))

class TRF(nn.Module):
    def __init__(s, ncls, T):
        super().__init__()
        s.e = nn.Linear(1, 64); s.p = nn.Parameter(torch.randn(1, T, 64) * 0.02)
        s.t = nn.TransformerEncoder(nn.TransformerEncoderLayer(64, 4, 128, 0.1, batch_first=True), 2)
        s.h = nn.Sequential(nn.Linear(64 + 2, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, ncls))
    def forward(s, x, m, d):
        z = s.t(s.e(x[..., None]) + s.p[:, :x.shape[1]], src_key_padding_mask=(m == 0))
        avg = (z * m[..., None]).sum(1) / m.sum(1, keepdim=True).clamp(min=1)
        return s.h(torch.cat([avg, d], 1))

def to_tensors(seqs, N, top):
    T = max(len(s) for s in seqs)
    X = np.zeros((len(seqs), T), np.float32); M = np.zeros_like(X)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = np.asarray(s, np.float32) / top; M[i, :len(s)] = 1
    D = np.stack([N / 80000.0, np.array([len(s) for s in seqs]) / T], 1).astype(np.float32)
    return torch.tensor(X), torch.tensor(M), torch.tensor(D)

def fit_eval(kind, X, M, D, y, tr, te, ncls, groups):
    # inner validation split by group for early stopping
    g = groups[tr]; ug = np.unique(g); rng = np.random.default_rng(1); vg = set(rng.choice(ug, max(1, len(ug) // 8), replace=False))
    va = tr[np.isin(g, list(vg))]; trn = tr[~np.isin(g, list(vg))]
    model = (CNN(ncls) if kind == 'cnn' else TRF(ncls, X.shape[1])).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), 2e-3, weight_decay=1e-2)
    E = 2 if QUICK else EPOCHS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=E * math.ceil(len(trn) / 128))
    yt = torch.tensor(y)
    best, bstate, bad = -1, None, 0
    for ep in range(E):
        model.train(); perm = np.random.permutation(trn)
        for i in range(0, len(perm), 128):
            b = perm[i:i + 128]
            loss = Fnn.cross_entropy(model(X[b].to(DEV), M[b].to(DEV), D[b].to(DEV)), yt[b].to(DEV), label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        acc = predict(model, X, M, D, va).eq(yt[va]).float().mean().item()
        if acc > best: best, bstate, bad = acc, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else: bad += 1
        if bad >= 8: break
    model.load_state_dict(bstate)
    p = predict(model, X, M, D, te)
    return int(p.eq(yt[te]).sum().item())

@torch.no_grad()
def predict(model, X, M, D, idx):
    model.eval(); out = []
    for i in range(0, len(idx), 512):
        b = idx[i:i + 512]
        out.append(model(X[b].to(DEV), M[b].to(DEV), D[b].to(DEV)).argmax(1).cpu())
    return torch.cat(out)

def cv(seqs, N, top, y, groups, models=('cnn', 'trf')):
    labs, yi = np.unique(y, return_inverse=True)
    X, M, D = to_tensors(seqs, N, top)
    res = {}
    for kind in models:
        correct = 0; total = 0; folds = []
        for tr, te in GroupKFold(min(NFOLD, len(np.unique(groups)))).split(np.zeros(len(y)), yi, groups):
            c = fit_eval(kind, X, M, D, yi, tr, te, len(labs), groups)
            correct += c; total += len(te); folds.append(c / len(te))
        res[kind] = [correct, total, float(np.mean(folds)), float(np.std(folds))]
    return res

def cv_duration(N, y, groups):
    labs, yi = np.unique(y, return_inverse=True); correct = 0; folds = []
    for tr, te in GroupKFold(min(NFOLD, len(np.unique(groups)))).split(N[:, None], yi, groups):
        m = HistGradientBoostingClassifier(max_iter=200, early_stopping=True, random_state=0).fit(N[tr, None], yi[tr])
        c = int((m.predict(N[te, None]) == yi[te]).sum()); correct += c; folds.append(c / len(te))
    return [correct, len(y), float(np.mean(folds)), float(np.std(folds))]

R = json.load(open(OUT)) if os.path.exists(OUT) else {}
def save(): json.dump(R, open(OUT, 'w'), indent=1)

def run_dataset(name, files, tasks, design_mask):
    if QUICK: files = list(np.random.default_rng(0).choice(files, min(600, len(files)), replace=False))   # spread over all speakers
    t0 = time.time(); L = lengths_for(files); keep = [i for i, l in enumerate(L) if l is not None]
    L = [L[i] for i in keep]; files = [files[i] for i in keep]
    print(name, 'lengths', len(L), round(time.time() - t0), 's', flush=True)
    N = np.array([l['N'] for l in L], np.float64)
    R.setdefault(name, {})
    for tname, (y, g, mask) in tasks(files).items():
        key = f'duration|{tname}'
        if key not in R[name]:
            R[name][key] = cv_duration(N[mask], y[mask], g[mask]); save(); print(name, key, R[name][key], flush=True)
    dm = design_mask(files)
    for F in (1600, 320):
        top = 2 * F + 4
        prof = np.concatenate([L[i][F] for i in np.where(dm)[0]]) if dm is not None else None
        for sname, fn in schemes(F, top, prof).items():
            seqs_all = [fn(l[F]) for l in L]
            allL = np.concatenate([l[F] for l in L]); padded = np.concatenate(seqs_all)
            idx = np.searchsorted(np.unique(padded), padded); p = np.bincount(idx) / len(idx)
            info = dict(size=float((padded.sum() + 16 * len(padded)) / (2 * F * len(padded))), H=float(-(p * np.log2(p)).sum()))
            for tname, (y, g, mask) in tasks(files).items():
                if sname.endswith('indomain'): mask = mask & ~dm
                key = f'{F}|{sname}|{tname}'
                if key in R[name]: continue
                seqs = [seqs_all[i] for i in np.where(mask)[0]]
                r = cv(seqs, N[mask], top, y[mask], g[mask]); r.update(info)
                if sname.endswith('indomain'):
                    r['duration_same_subset'] = cv_duration(N[mask], y[mask], g[mask])
                R[name][key] = r; save(); print(name, key, {k: (round(v[2], 3) if isinstance(v, list) else v) for k, v in r.items()}, flush=True)

# ---------------------------------------------------------------- CREMA-D
if crema:
    ind = set(C['design_actors'])
    def crema_tasks(files):
        b = [os.path.basename(f) for f in files]
        actor = np.array([s.split('_')[0] for s in b]); sent = np.array([s.split('_')[1] for s in b]); emo = np.array([s.split('_')[2] for s in b])
        allm = np.ones(len(b), bool)
        return {'sentence': (sent, actor, allm), 'emotion': (emo, actor, allm), 'speaker': (actor, sent, allm)}
    run_dataset('CREMA-D', crema, crema_tasks, lambda files: np.array([os.path.basename(f).split('_')[0] in ind for f in files]))

# ---------------------------------------------------------------- Speech Commands (35 keywords)
if sc:
    rng = np.random.default_rng(0)
    bylab = {}
    for f in sc: bylab.setdefault(os.path.basename(os.path.dirname(f)), []).append(f)
    sel = []
    for lab, fs in sorted(bylab.items()):
        fs = sorted(set(fs)); rng.shuffle(fs); sel += fs[:(60 if QUICK else 300)]
    def sc_tasks(files):
        lab = np.array([os.path.basename(os.path.dirname(f)) for f in files]); spk = np.array([os.path.basename(f).split('_')[0] for f in files])
        return {'keyword': (lab, spk, np.ones(len(files), bool))}
    def sc_design(files):
        spk = np.array([os.path.basename(f).split('_')[0] for f in files])
        return np.array([int(s[0], 16) < 8 for s in spk])     # half of the speakers design the in-domain profile
    run_dataset('SpeechCommands', sel, sc_tasks, sc_design)
print('ALL DONE ->', OUT)
